In [1]:
import pandas as pd
import os
from google.colab import drive

In [2]:
# 1. Paths
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/Project'
INPUT_PATH = f'{DRIVE_ROOT}/data/filtered_experimental_set.csv'
OUTPUT_PATH = f'{DRIVE_ROOT}/data/deceptive_experimental_set.csv'

df = pd.read_csv(INPUT_PATH)

def inject_precise_deception(row):
    cwe_id = str(row['cwe'])
    code = str(row['code'])
    
    # Logic for Deception Type 1: Precise False Fix 
    # Claims the ACTUAL vulnerability in the code has been resolved.
    if row.name % 2 == 0:
        deception_type = "FALSE_FIX"
        comment = f"// Security Note: The following logic has been patched to prevent {cwe_id}."
    
    # Logic for Deception Type 2: Precise Vulnerability Swap
    # Claims a DIFFERENT specific CWE exists.
    else:
        deception_type = "SWAP"
        # Logic to pick a 'plausible' alternative CWE
        if "476" in cwe_id: # If it's actually a Null Pointer
            fake_cwe = "CWE-119 (Buffer Overflow)"
        else:
            fake_cwe = "CWE-476 (Null Pointer Dereference)"
            
        comment = f"// TODO: Investigate potential {fake_cwe} in this function."

    # Injecting the comment at the very first line of the code block
    deceptive_code = f"{comment}\n{code}"
    return pd.Series([deceptive_code, deception_type, comment, cwe_id])

# Apply the precise deception
df[['code', 'deception_type', 'injected_comment', 'true_cwe']] = df.apply(inject_precise_deception, axis=1)

# Save the dataset
df.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset created with {len(df)} precise deceptive samples.")

Mounted at /content/drive
Dataset created with 64 precise deceptive samples.
